# Laya-MLX Demonstration Notebook

This notebook demonstrates the basics to advanced usage of `laya-mlx`, a library for running native MLX inference for Laya typed decision models on Apple Silicon.

## 1. Setup

First, let's install the library.

In [ ]:
!pip install laya-mlx

Note: If you want to use the demo features, use `!pip install 'laya-mlx[demo]'`.

## 2. Basics: Simple Decision

Let's load the default English agent and make a simple choice prediction.

In [ ]:
import laya_mlx as laya

# Load the model. This will download the weights if not cached.
agent = laya.load("aac6fef/laya-mlx")

# We want the agent to categorize the department based on a customer message.
result = agent.predict(
    "I was billed twice. Please refund the duplicate.",
    {
        "department": {
            "type": "choice",
            "instructions": "Who should handle this?",
            "criteria": ["billing", "technical", "sales"],
        }
    },
)

print("Selected Department:", result["answers"]["department"])
print("Full Result:")
import json
print(json.dumps(result, indent=2, ensure_ascii=False))

## 3. Advanced Field Extraction

Laya allows extracting multiple typed fields at once, including `choice`, `score`, and `noul` (boolean).

In [ ]:
result_advanced = agent.predict(
    "I was billed twice. Please refund the duplicate today.",
    {
        "department": {
            "type": "choice",
            "instructions": "Which team should handle this request?",
            "criteria": {
                "billing": "invoices, payments, refunds",
                "technical": "bugs and outages",
                "sales": "new purchases",
            },
        },
        "urgency": {
            "type": "score",
            "instructions": "How urgent is this request?",
            "criteria": ["not urgent", "soon", "critical"],
        },
        "refund": {
            "type": "noul",
            "instructions": "Does the customer ask for money back?",
        },
    },
)

print("Advanced Extraction Answers:")
print(json.dumps(result_advanced["answers"], indent=2, ensure_ascii=False))

## 4. Language Routing

For multilingual support, you can use the `Router` to automatically direct requests to the appropriate model (e.g., English vs Multilingual).

In [ ]:
from laya_mlx import Router, triage_questions

# The Router can keep multiple models loaded or load them on demand.
# Let's set max_loaded to 2 to keep both English and Multilingual ready if needed.
router = Router(dtype="float16", max_loaded=2)

# Pass a Chinese message to the router
result_routing = router.predict({"message": "发票被重复扣款，请退款。"}, triage_questions())

print("Routing Information:", result_routing.get("routing"))
print("Routing Answers:")
print(json.dumps(result_routing.get("answers", {}), indent=2, ensure_ascii=False))

## 5. Shortlisting Large Choice Sets

If you have hundreds of choices, you might run out of token budget. `predict_shortlist` uses embeddings to narrow down choices before running the decision model.

In [ ]:
# Using the same agent loaded earlier
embed_fn = laya.embed_fn_from_agent(agent)  # Extracts an embedder from the loaded encoder

large_criteria = [
    "billing", "technical", "sales", "hr", "legal", 
    "facilities", "marketing", "pr", "engineering", "design",
    "product", "support", "operations", "finance", "security"
]

result_shortlist = laya.predict_shortlist(
    agent, 
    "My computer screen is completely black.", 
    {
        "department": {
            "type": "choice",
            "instructions": "Who should handle this?",
            "criteria": large_criteria
        }
    },
    embed_fn, 
    k=5 # Keep only the top 5 most relevant choices based on embedding similarity
)

print("Shortlist information (Top 5 kept labels and scores):")
print(json.dumps(result_shortlist.get("shortlist"), indent=2, ensure_ascii=False))
print("\nFinal Selected Department:", result_shortlist["answers"]["department"])